In [2]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
import re

## 1. RNN-Based Text Generation

In [44]:
# Dataset:
import requests

url = "https://www.gutenberg.org/files/11/11-0.txt"
raw_text = requests.get(url).text
raw_text = re.sub(r'\s+', ' ', raw_text)  # Replace multiple whitespace with a single space
raw_text = raw_text.strip()  # Remove leading and trailing whitespace
raw_text = raw_text[:10000]  # Use only the first 10,000 characters for this example
print(len(raw_text), raw_text[:500])

specials = ["<pad>", "<sos>", "<eos>", "<unk>"]
end_punctuations = [".", "!", "?", "!?", "...", "?!"] # Common sentence-ending punctuation marks

# Tokenization (with sentence boundaries):
def tokenize_with_sentence_boundaries(text):
    base_tokens = re.findall(r"\w+|[^\w\s*]", text)
    tokens = ["<sos>"]
    for token in base_tokens:
        tokens.append(token)
        if token in end_punctuations:
            tokens.append("<eos>")
            tokens.append("<sos>")
    if tokens and tokens[-1] == "<sos>":
            tokens.pop()
    if tokens and tokens[-1] != "<eos>":
            tokens.append("<eos>")
    return tokens


processed_text = tokenize_with_sentence_boundaries(raw_text) 
print(f"Number of tokens: {len(processed_text)}, First 500 characters: {processed_text[:500]}")

def encoder(tokens):
    vocab = sorted(set(specials + tokens))
    vocab = {token: idx for idx , token in enumerate(vocab)}
    encoded = [vocab.get(token, vocab["<unk>"]) for token in tokens]
    return encoded, vocab


encoded_text, vocab = encoder(processed_text)
print(f"Vocabulary size: {len(vocab)}, \nFirst 100 tokens in vocabulary: {list(vocab.keys())[:100]} , \nFirst 100 encoded tokens: {encoded_text[:100]}")

# Dataset and DataLoader:
class TextDataset(Dataset):
    def __init__(self, encoded_text, seq_length):
        self.encoded_text = encoded_text
        self.seq_length = seq_length

    def __len__(self):
        return len(self.encoded_text) - self.seq_length

    def __getitem__(self, idx):
        input_seq = self.encoded_text[idx:idx + self.seq_length]
        target_seq = self.encoded_text[idx + 1:idx + self.seq_length + 1]
        return torch.tensor(input_seq), torch.tensor(target_seq)
    
dataset = TextDataset(encoded_text, seq_length=50)
dataloader = DataLoader(dataset, batch_size=32, shuffle=True)



10000 *** START OF THE PROJECT GUTENBERG EBOOK 11 *** [Illustration] Alice’s Adventures in Wonderland by Lewis Carroll THE MILLENNIUM FULCRUM EDITION 3.0 Contents CHAPTER I. Down the Rabbit-Hole CHAPTER II. The Pool of Tears CHAPTER III. A Caucus-Race and a Long Tale CHAPTER IV. The Rabbit Sends in a Little Bill CHAPTER V. Advice from a Caterpillar CHAPTER VI. Pig and Pepper CHAPTER VII. A Mad Tea-Party CHAPTER VIII. The Queen’s Croquet-Ground CHAPTER IX. The Mock Turtle’s Story CHAPTER X. The Lobster
Number of tokens: 2509, First 500 characters: ['<sos>', 'START', 'OF', 'THE', 'PROJECT', 'GUTENBERG', 'EBOOK', '11', '[', 'Illustration', ']', 'Alice', '’', 's', 'Adventures', 'in', 'Wonderland', 'by', 'Lewis', 'Carroll', 'THE', 'MILLENNIUM', 'FULCRUM', 'EDITION', '3', '.', '<eos>', '<sos>', '0', 'Contents', 'CHAPTER', 'I', '.', '<eos>', '<sos>', 'Down', 'the', 'Rabbit', '-', 'Hole', 'CHAPTER', 'II', '.', '<eos>', '<sos>', 'The', 'Pool', 'of', 'Tears', 'CHAPTER', 'III', '.', '<eos>', '<so

This: 144696 *** start of the project gutenberg ebook 11 ***

[illustration]




alice’s adventures in wonderland

by lewis carroll

the millennium fulcrum edition 3.0

contents

 chapter i.     down the rabbit-hole
 chapter ii.    the pool of tears
 chapter iii.   a caucus-race and a long tale
 chapter iv.    the rabbit sends in a little bill
 chapter v.     advice from a caterpillar
 chapter vi.    pig and pepper
 chapter vii.   a mad tea-party
 chapter viii.  the queen’s croquet-ground
 chapter ix.    the


In [ ]:
# Model definition:

class TextGenerationModel(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim):
        super(TextGenerationModel, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.rnn = nn.LSTM(embedding_dim, hidden_dim, bidirectional=True, batch_first=True)
        self.fc = nn.Linear(hidden_dim * 2, vocab_size)  # *2 because of bidirectional
    
    def forward(self, x):
        embedded = self.embedding(x)
        output, _ = self.rnn(embedded)
        output = self.fc(output)
        return output